# 13 — v27 by terrain class and time of day (+2 h lead)

Performance of **v27** across the four terrain classes, split into four 6-hour
UTC buckets, at a **fixed +2 h lead**. Fixing the lead removes the largest
confound: error grows steeply with horizon, so anything averaged over leads is
mostly a statement about the lead mix.

Evaluated at **mr0.50** with masked and visible stations kept apart, because
the two answer different questions:

* **visible** — how hard is this site to *forecast* when its own history is
  available?
* **masked** — how hard is it to *reconstruct* from neighbours alone?
* **penalty** = masked − visible — the cost of hiding the station, which is the
  quantity terrain should genuinely drive.

**The designed contrast** is `valley floor` (n=45, median 482 m) against
`elevated enclosed` (n=33, median 1408 m). Both are enclosed — median
normalised TPI −1.16 and −1.39, essentially the same shape — and differ almost
only in height, straddling the ~900 m winter inversion cap. Crossed with time
of day, that becomes a sharp test:

* penalty on valley floors worst **at night** (18-06 UTC) and not on elevated
  enclosed sites → **cold-air pooling**. At night a valley floor decouples from
  the regional field, so neighbours stop being informative.
* penalty equally bad in both classes, or flat across the day → **enclosure
  geometry**: a basin is poorly constrained by ridge stations regardless of
  hour or altitude.

Buckets are **UTC** (Swiss solar time ≈ UTC+0:32) and taken from the **target**
time. At a fixed lead the 90-min origin stride populates 16 of 24 UTC hours,
exactly 4 per bucket, so the four groups are evenly sampled.

In [ ]:
# ── Bootstrap ────────────────────────────────────────────────────────────────
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks", "analysis")):
    if os.path.isfile(os.path.join(_c, "common.py")):
        if _c not in sys.path: sys.path.insert(0, _c)
        break
import importlib
import common as C
importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUN, MR, LEAD_MIN = "v27", "mr0.50", 120
TOD  = ["00-06", "06-12", "12-18", "18-24"]
CLS  = ["valley floor", "elevated enclosed", "open / slope",
        "exposed ridge/summit"]
CCOL = {"valley floor": "#C4502A", "elevated enclosed": "#E1A730",
        "open / slope": "#2E7D8C", "exposed ridge/summit": "#6A4C93"}

stn = C.station_table(); ns = C.norm_stats()
VARS, STD = ns["var_names"], ns["std"]
KEEP = C.keep_mask(stn, VARS); N = len(stn)
CLS_IDX = np.array([CLS.index(c) for c in stn.terrain_class])
print(f"{RUN} @ {MR}, +{LEAD_MIN} min")
print(stn.terrain_class.value_counts().reindex(CLS).to_string())

## 1. One streaming pass

Accumulates absolute error at **(day, time-of-day, terrain class, variable)**
for masked stations, visible stations and persistence. The day axis exists so
section 4 can bootstrap over days: consecutive 90-minute windows are strongly
dependent, so a window-level test would badly overstate significance.

In [ ]:
CACHE_F = os.path.join(C.CACHE, f"n13_{RUN}_{MR}_lead{LEAD_MIN}.npz")
def build(force=False):
    if os.path.isfile(CACHE_F) and not force:
        z = np.load(CACHE_F); print(f"loaded cache ({int(z['n_days'])} days)")
        return {k: z[k] for k in z.files}
    d = C.load_dump(RUN, MR)
    P, T, M, MI = d["preds"], d["targets"], d["masks"], d["masked_idx"]
    TH = d["target_hours"]; grid = d["delta_steps"][0].numpy().astype(int)
    k = int(np.where(grid * 10 == LEAD_MIN)[0][0])
    Mw, V = P.shape[0], len(VARS)
    th_all = TH[:, k].numpy().astype(np.float64)
    day_all = np.floor(th_all / 24.0).astype(np.int64)
    day_all -= day_all.min(); ND = int(day_all.max()) + 1
    A = {f"{s}_{q}": np.zeros((ND, 4, 4, V))
         for s in ("msk", "vis", "per") for q in ("sum", "cnt")}
    CH = 1000
    for a in range(0, Mw, CH):
        b = min(a + CH, Mw)
        p = P[a:b, k].numpy().astype(np.float64)
        t = T[a:b, k, :, :V].numpy().astype(np.float64)
        t0 = T[a:b, 0, :, :V].numpy().astype(np.float64)      # persistence
        m = (M[a:b, k, :, :V].numpy() > 0.5) & KEEP[None]
        m0 = M[a:b, 0, :, :V].numpy() > 0.5
        e_mod = np.abs(p - t) * STD[None]
        e_per = np.abs(t0 - t) * STD[None]
        mi = MI[a:b].numpy()
        sel = np.zeros((b - a, N), bool)
        np.put_along_axis(sel, mi, True, axis=1)
        tod = ((th_all[a:b] % 24) // 6).astype(np.int64)
        day = day_all[a:b]
        d_ax = np.broadcast_to(day[:, None, None], e_mod.shape)
        t_ax = np.broadcast_to(tod[:, None, None], e_mod.shape)
        c_ax = np.broadcast_to(CLS_IDX[None, :, None], e_mod.shape)
        v_ax = np.broadcast_to(np.arange(V)[None, None, :], e_mod.shape)
        for tag, w, err in (("msk", m & sel[:, :, None], e_mod),
                            ("vis", m & ~sel[:, :, None], e_mod),
                            ("per", m & m0, e_per)):
            idx = (d_ax[w], t_ax[w], c_ax[w], v_ax[w])
            np.add.at(A[f"{tag}_sum"], idx, err[w])
            np.add.at(A[f"{tag}_cnt"], idx, 1.0)
    A["n_days"] = np.array(ND)
    np.savez_compressed(CACHE_F, **A)
    print(f"streamed {Mw:,} windows, lead index {k}, {ND} days")
    return A

A = build()
def mae(tag, day=slice(None)):
    s = A[f"{tag}_sum"][day].sum(0); c = A[f"{tag}_cnt"][day].sum(0)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan), c
Mm, Cm = mae("msk"); Mv, Cv = mae("vis"); Mp, Cp = mae("per")
print("\nvalid predictions per (class, bucket), summed over variables:")
print(pd.DataFrame(Cm.sum(2).T.astype(int), index=CLS, columns=TOD).to_string())

# ── Free dump tensors ──
del d, P, T, M, MI, TH
import gc; gc.collect()


## 2. Error by terrain class and time of day

Physical units. Masked and visible side by side, per variable.

In [ ]:
fig, axes = plt.subplots(2, len(VARS), figsize=(3.6*len(VARS), 6.4),
                         sharex=True)
for ri, (tag, arr, lab) in enumerate((("vis", Mv, "VISIBLE"),
                                      ("msk", Mm, "MASKED"))):
    for ci, v in enumerate(VARS):
        ax = axes[ri, ci]
        for k_, c in enumerate(CLS):
            ax.plot(range(4), arr[:, k_, ci], "o-", ms=4, lw=1.5,
                    color=CCOL[c], label=c)
        ax.set_xticks(range(4)); ax.set_xticklabels(TOD, rotation=45,
                                                    ha="right", fontsize=7.5)
        ax.grid(alpha=.3)
        if ri == 0: ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if ci == 0: ax.set_ylabel(f"{lab}\nMAE", fontsize=10)
# one scale per variable so the two rows are comparable
for ci in range(len(VARS)):
    lo = np.nanmin([Mv[:, :, ci].min(), Mm[:, :, ci].min()])
    hi = np.nanmax([Mv[:, :, ci].max(), Mm[:, :, ci].max()])
    pad = .06*(hi-lo)
    for ri in range(2): axes[ri, ci].set_ylim(lo-pad, hi+pad)
axes[0, -1].legend(fontsize=7)
fig.suptitle(f"{RUN} @ {MR}, +{LEAD_MIN}min — MAE by terrain class and "
             f"time of day (UTC, target time)", y=1.01)
# ── Sync y-axis per variable column across rows ──
for vi in range(len(VARS)):
    col_axes = [axes[ri, vi] for ri in range(2)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
plt.tight_layout(); C.save_fig(fig, "13_mae_class_tod"); plt.show()

## 3. Masking penalty — the headline

`MAE(masked) − MAE(visible)`, per class and bucket. This is the cost of hiding
a station, so it isolates reconstructability from intrinsic site difficulty.

In [ ]:
PEN = Mm - Mv
fig, axes = plt.subplots(1, len(VARS), figsize=(3.6*len(VARS), 3.6))
for ci, v in enumerate(VARS):
    ax = axes[ci]
    for k_, c in enumerate(CLS):
        ax.plot(range(4), PEN[:, k_, ci], "o-", ms=5, lw=1.8,
                color=CCOL[c], label=c)
    ax.axhline(0, color="k", lw=.9)
    ax.set_xticks(range(4)); ax.set_xticklabels(TOD, rotation=45,
                                                ha="right", fontsize=7.5)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel("masking penalty\nMAE(masked) − MAE(visible)", fontsize=9)
axes[-1].legend(fontsize=7)
fig.suptitle(f"{RUN} — cost of hiding a station, by terrain class and "
             f"time of day (+{LEAD_MIN}min)", y=1.05)
plt.tight_layout(); C.save_fig(fig, "13_masking_penalty_class_tod"); plt.show()

tab = pd.DataFrame({(c, v): PEN[:, k_, ci] for k_, c in enumerate(CLS)
                    for ci, v in enumerate(VARS)}, index=TOD)
tab.columns.names = ["terrain class", "variable"]
display(tab.round(3).style.background_gradient(cmap="OrRd")
        .set_caption("masking penalty (physical units); higher = harder to "
                     "reconstruct from neighbours"))
C.save_table(tab, "13_masking_penalty_class_tod")

## 4. The designed contrast, with confidence intervals

`valley floor` vs `elevated enclosed` — same shape, different side of the
inversion cap. Resampling **days** with replacement, because consecutive
windows are far from independent. If the difference in penalty is significant
at night and not during the day, the mechanism is cold-air pooling rather than
enclosure geometry.

In [ ]:
def boot_diff(cls_a, cls_b, vi, n_boot=2000, seed=0):
    ia, ib = CLS.index(cls_a), CLS.index(cls_b)
    used = (A["msk_cnt"].sum((1, 2, 3)) + A["vis_cnt"].sum((1, 2, 3))) > 0
    sm, cm = A["msk_sum"][used], A["msk_cnt"][used]
    sv, cv = A["vis_sum"][used], A["vis_cnt"][used]
    ND = sm.shape[0]; rng = np.random.default_rng(seed)
    out = np.empty((n_boot, 4))
    for i in range(n_boot):
        t_ = rng.integers(0, ND, ND)
        pa = (sm[t_][:, :, ia, vi].sum(0)/np.maximum(cm[t_][:, :, ia, vi].sum(0), 1)
              - sv[t_][:, :, ia, vi].sum(0)/np.maximum(cv[t_][:, :, ia, vi].sum(0), 1))
        pb = (sm[t_][:, :, ib, vi].sum(0)/np.maximum(cm[t_][:, :, ib, vi].sum(0), 1)
              - sv[t_][:, :, ib, vi].sum(0)/np.maximum(cv[t_][:, :, ib, vi].sum(0), 1))
        out[i] = pa - pb
    return out, ND

rows = []
fig, axes = plt.subplots(1, len(VARS), figsize=(3.6*len(VARS), 3.4))
for ci, v in enumerate(VARS):
    b, ND = boot_diff("valley floor", "elevated enclosed", ci)
    obs = PEN[:, CLS.index("valley floor"), ci] - \
          PEN[:, CLS.index("elevated enclosed"), ci]
    lo, hi = np.nanpercentile(b, [2.5, 97.5], axis=0)
    ax = axes[ci]
    ax.errorbar(range(4), obs, yerr=[obs-lo, hi-obs], fmt="o", ms=5, lw=1.4,
                color="#C4502A")
    sig = (lo > 0) | (hi < 0)
    ax.plot(np.arange(4)[sig], obs[sig], "o", ms=9, mfc="none", mec="#C4502A",
            mew=2)
    ax.axhline(0, color="k", lw=.9); ax.grid(alpha=.3)
    ax.set_xticks(range(4)); ax.set_xticklabels(TOD, rotation=45, ha="right",
                                                fontsize=7.5)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    for t_ in range(4):
        rows.append({"variable": v, "bucket": TOD[t_], "diff": obs[t_],
                     "lo": lo[t_], "hi": hi[t_], "significant": bool(sig[t_])})
axes[0].set_ylabel("penalty difference\nvalley floor − elevated enclosed",
                   fontsize=9)
fig.suptitle(f"Same shape, opposite sides of the inversion cap — "
             f"95% CI from {ND}-day block bootstrap  (ringed = CI excludes 0)",
             y=1.06)
plt.tight_layout(); C.save_fig(fig, "13_valley_vs_elevated_ci"); plt.show()
res = pd.DataFrame(rows)
display(res.round(3).style.apply(
    lambda s: ["background-color:#ffe9e0" if x else "" for x in res.significant],
    axis=0).set_caption("positive = valley floors are harder to reconstruct "
                        "than equally-enclosed sites above the inversion cap"))
C.save_table(res, "13_valley_vs_elevated_bootstrap")

## 5. Skill against persistence, per class and bucket

`1 − MAE(model) / MAE(persistence)`, using the model's **visible** stations so
the comparison is like-for-like — persistence also needs the station's own
history. Where skill collapses, the model is adding little over "nothing
changes".

In [ ]:
SK = 1 - Mv / np.where(Mp > 0, Mp, np.nan)
fig, axes = plt.subplots(1, len(VARS), figsize=(3.6*len(VARS), 3.4), sharey=True)
for ci, v in enumerate(VARS):
    ax = axes[ci]
    for k_, c in enumerate(CLS):
        ax.plot(range(4), SK[:, k_, ci], "o-", ms=5, lw=1.6, color=CCOL[c],
                label=c)
    ax.axhline(0, color="k", lw=.9); ax.grid(alpha=.3)
    ax.set_xticks(range(4)); ax.set_xticklabels(TOD, rotation=45, ha="right",
                                                fontsize=7.5)
    ax.set_title(f"{v}", fontsize=10)
axes[0].set_ylabel("skill vs persistence"); axes[-1].legend(fontsize=7)
fig.suptitle(f"{RUN} visible stations — skill vs persistence by class and "
             f"time of day (+{LEAD_MIN}min)", y=1.05)
plt.tight_layout(); C.save_fig(fig, "13_skill_class_tod"); plt.show()
sk = pd.DataFrame({(c, v): SK[:, k_, ci] for k_, c in enumerate(CLS)
                   for ci, v in enumerate(VARS)}, index=TOD)
sk.columns.names = ["terrain class", "variable"]
display(sk.round(3).style.background_gradient(cmap="RdYlGn", vmin=-.2, vmax=.7))
C.save_table(sk, "13_skill_class_tod")

## Interpretation

**Read section 4 first.** It is the only panel with error bars, and it decides
between the two mechanisms. Significance at night only ⇒ **cold-air pooling**:
valley floors decouple after sunset and their neighbours stop being
informative. Significance flat across the day, or none at all ⇒ **enclosure
geometry**, which is altitude-independent — a basin is poorly constrained by
ridge stations at any hour.

**Section 3 without section 4 is not evidence.** Penalty differences of a few
hundredths of a degree between classes will look convincing on a line plot and
may sit well inside the day-block CI. Quote the interval, not the point.

**Section 2's two rows must be read against each other.** A class that is bad
in *both* rows is intrinsically hard (high local variance). A class bad only in
the masked row is hard *to reconstruct* — that is the spatial-context claim,
and it is what section 3 isolates.

**Expected, and worth checking rather than assuming:** temperature and humidity
should show the strongest diurnal structure, since both are radiatively driven;
pressure should be nearly flat across buckets in every class, and if it is not,
suspect the bucketing rather than the physics.

**Caveats.** Terrain classes correlate with altitude by construction, and
`elevated enclosed` is 28/33 Alpine while `valley floor` draws from the
Plateau, Alps and Jura — so the contrast carries a mild regional confound
alongside the elevation one. Restricting both classes to Alpine stations
(12 vs 18) removes it at the cost of sample size, and is the right robustness
check if this becomes a headline result.